# Walmart Demand Forecasting
## Notebook 05: Machine Learning Models
## Objectives
Develop and evaluate machine learning models to forecast Walmart's weekly sales and determine whether they outperform the established baseline forecasting methods.
### Dataset
- walmart_features.csv
### Tasks
- Load the processed feature dataset.
- Split the data using a time-based train-test strategy.
- Prepare features and target variables for model training.
- Train multiple machine learning models.
- Evaluate model performance using MAE and RMSE.
- Compare machine learning models with the baseline forecasting model.
- Select the best-performing forecasting model.
### Expected Output
By the end of this notebook, we will develop and evaluate multiple machine learning models for weekly sales forecasting. Their performance will be compared against the baseline established in the previous notebook to determine whether machine learning provides meaningful improvements in forecasting accuracy.

#1. Load and check data

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

df = pd.read_csv(
    PROCESSED_PATH / "walmart_features.csv",
    parse_dates=["Date"]
)

df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,...,Size,Year,Month,Week,lag_1,lag_2,lag_4,lag_52,rolling_mean_4,rolling_std_4
0,1,1,2010-03-05,21827.90,46.50,2.625,0.0,0.0,0.0,0.0,...,151315,2010,3,9,19403.54,41595.55,24924.50,NaN,32990.7700,12832.106391
1,1,1,2010-03-12,21043.39,57.79,2.667,0.0,0.0,0.0,0.0,...,151315,2010,3,10,21827.90,19403.54,46039.49,NaN,32216.6200,13554.047185
2,1,1,2010-03-19,22136.64,54.58,2.720,0.0,0.0,0.0,0.0,...,151315,2010,3,11,21043.39,21827.90,41595.55,NaN,25967.5950,10467.484020
3,1,1,2010-03-26,26229.21,51.45,2.732,0.0,0.0,0.0,0.0,...,151315,2010,3,12,22136.64,21043.39,19403.54,NaN,21102.8675,1222.784968
4,1,1,2010-04-02,57258.43,62.27,2.719,0.0,0.0,0.0,0.0,...,151315,2010,4,13,26229.21,22136.64,21827.90,NaN,22809.2850,2325.929203


In [13]:
print(df.shape)

df.info()

(408436, 25)
<class 'pandas.DataFrame'>
RangeIndex: 408436 entries, 0 to 408435
Data columns (total 25 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Store           408436 non-null  int64         
 1   Dept            408436 non-null  int64         
 2   Date            408436 non-null  datetime64[us]
 3   Weekly_Sales    408436 non-null  float64       
 4   Temperature     408436 non-null  float64       
 5   Fuel_Price      408436 non-null  float64       
 6   MarkDown1       408436 non-null  float64       
 7   MarkDown2       408436 non-null  float64       
 8   MarkDown3       408436 non-null  float64       
 9   MarkDown4       408436 non-null  float64       
 10  MarkDown5       408436 non-null  float64       
 11  CPI             408436 non-null  float64       
 12  Unemployment    408436 non-null  float64       
 13  IsHoliday       408436 non-null  int64         
 14  Type            408436 non-null  s

In [14]:
df.describe()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,...,Size,Year,Month,Week,lag_1,lag_2,lag_4,lag_52,rolling_mean_4,rolling_std_4
count,408436.000000,408436.000000,408436,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,...,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,408436.000000,261083.000000,408436.000000,408436.000000
mean,22.191548,44.223861,2011-07-03 00:44:53.521628,16028.603006,60.803729,3.381032,2670.478451,906.570627,476.685936,1116.442782,...,136802.761801,2010.997987,6.580076,26.394821,16024.891012,16032.662405,16043.625770,16225.029406,16034.846179,1975.883447
min,1.000000,1.000000,2010-03-05 00:00:00,-4988.940000,-2.060000,2.513000,0.000000,-265.760000,-29.100000,0.000000,...,34875.000000,2010.000000,1.000000,1.000000,-4988.940000,-4988.940000,-4988.940000,-4988.940000,-989.500000,0.000000
25%,11.000000,18.000000,2010-11-05 00:00:00,2119.135000,47.920000,2.961000,0.000000,0.000000,0.000000,0.000000,...,93638.000000,2010.000000,4.000000,15.000000,2116.897500,2118.482500,2118.075000,2389.290000,2182.215000,270.279194
50%,22.000000,37.000000,2011-07-01 00:00:00,7651.315000,62.940000,3.480000,0.000000,0.000000,0.000000,0.000000,...,140167.000000,2011.000000,7.000000,26.000000,7648.325000,7657.040000,7663.510000,7998.550000,7766.253750,760.263509
75%,33.000000,74.000000,2012-03-02 00:00:00,20274.700000,74.710000,3.743000,3071.520000,5.500000,5.200000,489.960000,...,202505.000000,2012.000000,9.000000,38.000000,20264.295000,20275.550000,20286.882500,20520.920000,20371.000000,1931.910221
max,45.000000,99.000000,2012-10-26 00:00:00,693099.360000,100.140000,4.468000,88646.760000,104519.540000,141630.610000,67474.850000,...,219622.000000,2012.000000,12.000000,52.000000,693099.360000,693099.360000,693099.360000,693099.360000,339472.757500,280480.021441
std,12.783730,30.504301,NaN,22722.046276,18.133715,0.449805,6127.435073,5157.834484,5576.765603,3947.934964,...,60952.336997,0.790787,3.198616,13.958951,22723.608367,22731.853406,22754.401147,22534.761596,22230.357784,5143.792861


#2. Train/Test split

In [15]:
train_df = df[df["Date"] < "2012-01-01"].copy()

test_df = df[df["Date"] >= "2012-01-01"].copy()

print(train_df.shape)
print(test_df.shape)

(281140, 25)
(127296, 25)


#3. Define features and targets and check missing values

In [16]:
X_train = train_df.drop(columns=["Weekly_Sales", "Date"])

y_train = train_df["Weekly_Sales"]

X_test = test_df.drop(columns=["Weekly_Sales", "Date"])

y_test = test_df["Weekly_Sales"]

In [17]:
X_train.isna().sum().sort_values(ascending=False)

lag_52            144798
Store                  0
Type                   0
rolling_mean_4         0
lag_4                  0
lag_2                  0
lag_1                  0
Week                   0
Month                  0
Year                   0
Size                   0
IsHoliday              0
Dept                   0
Unemployment           0
CPI                    0
MarkDown5              0
MarkDown4              0
MarkDown3              0
MarkDown2              0
MarkDown1              0
Fuel_Price             0
Temperature            0
rolling_std_4          0
dtype: int64

In [18]:
X_test.isna().sum().sort_values(ascending=False)

lag_52            2555
Store                0
Type                 0
rolling_mean_4       0
lag_4                0
lag_2                0
lag_1                0
Week                 0
Month                0
Year                 0
Size                 0
IsHoliday            0
Dept                 0
Unemployment         0
CPI                  0
MarkDown5            0
MarkDown4            0
MarkDown3            0
MarkDown2            0
MarkDown1            0
Fuel_Price           0
Temperature          0
rolling_std_4        0
dtype: int64

### 4. Remove Seasonal Lag Feature

Although `lag_52` was created during feature engineering to capture annual seasonality, it contains a large number of missing values because one year of historical data is unavailable for many observations.

Since linear regression models in scikit-learn cannot handle missing values directly, `lag_52` is excluded from the initial machine learning models. This allows all available training observations to be retained without introducing potentially biased imputation.

The feature remains valuable for seasonal baseline forecasting (Notebook 04) and can be revisited in future model improvements.

In [19]:
X_train = X_train.drop(columns=["lag_52"])
X_test = X_test.drop(columns=["lag_52"])
print(X_train.shape)
print(X_test.shape)

print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

(281140, 22)
(127296, 22)
0
0


### 5. Feature Scaling

The numerical features have different value ranges (e.g., store size, CPI, lag features, and markdown variables). To ensure consistent feature scales and improve the performance of linear models, numerical features are standardized using `StandardScaler`.

The fitted scaler is learned from the training data only and then applied to both the training and test datasets to prevent data leakage.

In [22]:
numerical_features = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment",
    "Size",
    "Year",
    "Month",
    "Week",
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_std_4"
]

categorical_features = [
    "Store",
    "Dept",
    "IsHoliday",
    "Type"
]

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [25]:
linear_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Store','Dept','Temperature',...,'lag_4','rolling_mean_4','rolling_std_4']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of col

In [26]:
linear_pred = linear_pipeline.predict(X_test)